In [2]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import getpass
from datetime import date
import time


### Structure query

In [3]:
# set your API key before making the request
YOUR_API_KEY = getpass.getpass("Enter OpenET API Key Here:")
header = {"Authorization": YOUR_API_KEY}
# arguments to pass to API endpoint



Enter OpenET API Key Here: ········


### Request data

* Iterate through models and intervals
* Append model, eto source, interval, and date of extraction to standard files for queries to the `raster/timeseries/polygon`

# NEEDS TO BE UPDATED

In [14]:
from pathlib import Path
from urllib.request import urlretrieve
from datetime import date
import time
import io
import zipfile

models = [
    'ensemble', 'disalexi', 'eemetric', 'geesebal', 'ptjpl', 
    'sims', 'ssebop']
eto_source = ['gridMET','cimis']
intervals = ["daily"]#"monthly"]
today = date.today().strftime("%Y%m%d")
interval_dr = {
    "daily": ["2017-07-01", "2017-07-31"],
    "monthly": ["2017-01-01", "2017-12-31"]
}

for model in models:
    print(model)
    for interval in intervals:
        print(f"\tstarting {interval} extraction") 
        args = {
            "date_range": interval_dr[interval],
            "interval": interval,     # frequency of data: daily or monthly
            "asset_id": "projects/watrs-rfishman/assets/dwr/ca_delta_tw3",
            "model": model,     # Model options: Ensemble, geeSEBAL, SSEBop, SIMS, DisALEXI, PTJPL, or eeMETRIC.
            "overpass":True,        # Optional parameter for daily timesteps return only satellite overpass dates
            "variable": "ET",        # Variable options: ET, EToF, ETo, PR, NDVI
            "reference_et": eto_source[0], # ETo options: CIMiS, gridMET
            "units": "mm",           # Unit options: mm, in
            "file_format": "JSON",    # JSON, CSV
            "version": 2.1,
            "reducer": "mean"
        }
        # query the api
        resp = requests.post(
            headers=header,
            json=args,
            url="https://openet-api.org/raster/geotiff/stack"
        )
        output = resp.json()
        time.sleep(2)

        # Output will be dict k=time, v=link
        for k, v in resp.json().items():
            zipcontent = requests.get(v)
            zipmember = f"{model}_et_{k}.tif"
            zippath = Path("stack/" + zipmember)
            fn = f'stack/openet_standard_geotiff_stack_overpass_{interval}_{model}_{eto_source[0]}_{k}.tif'
            
            with zipfile.ZipFile(io.BytesIO(zipcontent.content)) as zip_ref:
                zip_ref.extract(zipmember, "stack/")
                zippath.rename(fn)
            
                print('\tfile saved to: \n\t\t'+ fn)


ensemble
	starting daily extraction
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_ensemble_gridMET_2017-07-08.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_ensemble_gridMET_2017-07-16.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_ensemble_gridMET_2017-07-24.tif
disalexi
	starting daily extraction
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_disalexi_gridMET_2017-07-08.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_disalexi_gridMET_2017-07-16.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_disalexi_gridMET_2017-07-24.tif
eemetric
	starting daily extraction
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_eemetric_gridMET_2017-07-08.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_eemetric_gridMET_2017-07-16.tif
	file saved to: 
		stack/openet_standard_geotiff_stack_overpass_daily_eemetr

## Inspect Standard Files to Verify relative to prior ground-based measurment comparison
# NEEDS TO BE UPDATES

In [1]:
# def assign_dt(df,datetime_col):
#     '''
#     inputs:
#     df: variable name of input dataframe [pandas dataframe]
#     datetime_col: column header for date time [string]
#     returns:
#     outdf: a dataframe with datetime_col set to index in datetime format
#     '''
#     outdf = df.copy()
#     outdf[datetime_col] = pd.to_datetime(outdf[datetime_col])
#     outdf.set_index(datetime_col,inplace=True)
#     return outdf

# # Read in US-TW3 dataset
# ground_data = pd.read_csv('ground_et_measurements/US-Tw3_daily_data.csv')
# ground_data = assign_dt(ground_data,'date')
# ground_data = ground_data.loc[(ground_data.index>='2016-01-01')&(ground_data.index<='2020-12-31')]
# # Assign date of reference dataset
# date_of_extraction = '20260916'

# # plot figures to verify reference dataset is of good quality
# for model in models[0:]:
#     mod_i_daily_df=pd.read_csv(f'point/openet_standard_point_daily_{model}_gridMET_{date_of_extraction}.csv')
#     mod_i_monthly_df=pd.read_csv(f'point/openet_standard_point_monthly_{model}_gridMET_{date_of_extraction}.csv')    
#     mod_i_daily_df = assign_dt(mod_i_daily_df,'time')
#     mod_i_monthly_df = assign_dt(mod_i_monthly_df,'time')
    
#     fig, axs = plt.subplots(2,1)
#     mod_i_daily_df.et.plot(ax=axs[0],label='OpenET',color='dodgerblue')
#     ground_data.ET.plot(ax=axs[0],label='US-TW3',style='.',ms=2,color='darkgray')
#     axs[0].set_ylabel('ET [mm]')
#     axs[0].set_xlabel('')
#     # axs[0].set_title('daily',x=0.8,y=0.8)
#     mod_i_daily_df.resample('ME').sum().et.plot(ax=axs[1],label='daily sum',color='dodgerblue')
#     ground_data.resample('ME').sum().ET.plot(ax=axs[1],label='US-TW3',style='.',ms=3,color='darkgray')
#     mod_i_monthly_df.et.plot(ax=axs[1],label='monthly',color='orange')
#     axs[1].set_ylabel('Total ET [mm]')
#     axs[1].set_xlabel('')
#     axs[1].legend(frameon=False,loc='upper center',ncol=2)
#     # axs[1].set_title('monthly',x=0.8,y=0.8)
#     fig.suptitle(model+ ' daily -> monthly Consistency Check',y=0.95)
#     plt.show()